# Eksplorasi Data (EDA) Skin Lesion — ISIC 2018 Task 3

Notebook EDA pendamping pipeline klasifikasi (`isic2018_resnet_pipeline.ipynb`). Semua analisis **read-only** (tidak training), hasil visual disimpan ke folder `output_eda/`. Tujuh tahap:

1. **Distribusi Kelas** — seberapa timpang antar 7 kelas diagnosis.
2. **Analisis `lesion_id`** — 1 image ≠ 1 lesi independen; cek `image_id` vs `lesion_id`, gambar per lesi, dan risiko **data leakage** saat split (lesi tercuil antar partition).
3. **Visualisasi Setiap Kelas** — 2–3 gambar/kelas untuk memahami dasar pembeda:
   shape, color, texture, border, size, global structure.
4. **Resolusi & Aspect Ratio** — distribusi lebar/tinggi/aspect ratio; apakah resize 224×224 masuk akal (cost, detail lesi, receptive field, memori, batch size).
5. **Distribusi Warna** — histogram RGB, brightness, contrast, saturation; termasuk sanity-check kuat-lemahnya ColorJitter agar augmentasi tidak menghancurkan warna asli.
6. **Duplikat / Near-Duplikat** — md5 (eksak) + dhash (near) antar partition, karena train ≈ validation bikin evaluasi bias.
7. **Ukuran Dataset Setelah Split** — jumlah **per kelas** (bukan hanya total) pada train/validation/test, plus implikasinya ke class weight, sampling, augmentasi, dan interpretasi F1 per kelas.

> Dataset mengikuti rilis resmi ISIC 2018 Challenge. Analisis `lesion_id` butuh `HAM10000_metadata.csv` (opsional) — otomatis terdeteksi di `/kaggle/input` atau folder `dataset/`; tanpa file itu tahap 2 tetap menampilkan penjelasan konsep lengkap.

## 1. Konfigurasi & Setup (path dataset + parameter EDA)

Path dataset **terdeteksi otomatis**: di Kaggle dari `/kaggle/input` (folder yang memuat `ISIC2018_Task3_Training_Input` dipilih otomatis), di lokal dari folder `dataset/`. Semua artefak EDA (CSV + PNG) disimpan ke `EDA_CFG['output_dir']`.

In [ ]:
# =========================================================================
# SECTION 1: KONFIGURASI & SETUP (path dataset + parameter EDA)
# =========================================================================
import os
import glob
import hashlib
import json
from pathlib import Path

EDA_CFG = {
    # ---- PATH DATASET (ikuti rilis resmi ISIC 2018 Task 3, seperti pipeline utama) ----
    "data_dir": "dataset",
    "train_img_dir": "ISIC2018_Task3_Training_Input",
    "test_img_dir":  "ISIC2018_Task3_Test_Input",
    "val_img_dir":   "ISIC2018_Task3_Validation_Input",
    "train_gt_dir":  "ISIC2018_Task3_Training_GroundTruth",
    "test_gt_dir":   "ISIC2018_Task3_Test_GroundTruth",
    "val_gt_dir":    "ISIC2018_Task3_Validation_GroundTruth",

    # ---- HAM10000 metadata (opsional, berisi kolom lesion_id) ----
    # Biarkan '' = auto-cari file bernama HAM10000_metadata.csv di /kaggle/input dan di dataset/.
    "ham10000_metadata": "",

    # ---- OUTPUT ----
    "output_dir": "output_eda",

    # ---- PARAMETER ANALISIS ----
    "class_columns": ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"],
    "num_classes": 7,
    "img_size": 224,              # ukuran input model yang direncanakan (untuk diskusi resize)
    "val_ratio": 0.15,            # proporsi validation dari training set (sama dengan pipeline utama)
    "random_state": 42,           # seed reproduksibilitas split & sampling
    "sample_size": 300,           # sampel gambar untuk EDA resolusi & aspect ratio
    "color_sample_size": 200,     # sampel gambar untuk EDA warna
    "near_dup_sample": 400,       # sampel gambar untuk deteksi near-duplicate (dhash)
    "full_hash": True,            # hash md5 SEMUA gambar (deteksi exact duplicate eksak); False = hanya sampel
    "ham_near_threshold": 8,      # jarak hamming maksimum dhash yang dianggap near-duplicate
}

# ---- Resolusi path (copy dari pipeline utama) ----
def _resolve_data_root(cfg):
    explicit = str(cfg["data_dir"])
    if os.path.isdir(os.path.join(explicit, cfg["train_img_dir"])):
        return explicit
    if os.path.isdir("/kaggle/input"):
        for entry in sorted(os.listdir("/kaggle/input")):
            cand = os.path.join("/kaggle/input", entry)
            if os.path.isdir(os.path.join(cand, cfg["train_img_dir"])):
                return cand
    return explicit

DATA_DIR = _resolve_data_root(EDA_CFG)

TRAIN_IMG_DIR = os.path.join(DATA_DIR, EDA_CFG["train_img_dir"])
TEST_IMG_DIR  = os.path.join(DATA_DIR, EDA_CFG["test_img_dir"])
VAL_IMG_DIR   = os.path.join(DATA_DIR, EDA_CFG["val_img_dir"])

def _first_csv(folder):
    hits = glob.glob(os.path.join(folder, "*.csv"))
    return hits[0] if hits else None

TRAIN_GT_PATH = _first_csv(os.path.join(DATA_DIR, EDA_CFG["train_gt_dir"]))
TEST_GT_PATH  = _first_csv(os.path.join(DATA_DIR, EDA_CFG["test_gt_dir"]))
VAL_GT_PATH   = _first_csv(os.path.join(DATA_DIR, EDA_CFG["val_gt_dir"]))

for p, d in [(TRAIN_IMG_DIR, "Training"), (TEST_IMG_DIR, "Test"), (VAL_IMG_DIR, "Validation")]:
    assert os.path.isdir(p), d + ": folder tidak ada -> " + p
for p, d in [(TRAIN_GT_PATH, "Train GT"), (TEST_GT_PATH, "Test GT"), (VAL_GT_PATH, "Val GT")]:
    assert p and os.path.isfile(p), d + ": file tidak ada -> " + str(p)

OUTPUT_DIR = Path(EDA_CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Cari HAM10000_metadata.csv (lesion_id) ----
def _find_ham_meta(cfg):
    candidates = []
    if cfg["ham10000_metadata"]:
        candidates.append(cfg["ham10000_metadata"])
    candidates.append(os.path.join(DATA_DIR, "HAM10000_metadata.csv"))
    if os.path.isdir("/kaggle/input"):
        for entry in sorted(os.listdir("/kaggle/input")):
            p = os.path.join("/kaggle/input", entry)
            if os.path.isfile(os.path.join(p, "HAM10000_metadata.csv")):
                candidates.append(os.path.join(p, "HAM10000_metadata.csv"))
    for c in candidates:
        if c and os.path.isfile(c):
            return c
    return None

HAM_META_PATH = _find_ham_meta(EDA_CFG)
CLASS_COLUMNS = EDA_CFG["class_columns"]

print("DATA_DIR      :", DATA_DIR)
print("OUTPUT_DIR    :", OUTPUT_DIR)
print("Train gambar  :", len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg"))))
print("Test gambar   :", len(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))))
print("Validation    :", len(glob.glob(os.path.join(VAL_IMG_DIR, "*.jpg"))))
print("HAM10000 meta :", HAM_META_PATH if HAM_META_PATH else "TIDAK DITEMUKAN -> analisis lesion_id terbatas")

## 2. Imports & Helper Umum

In [ ]:
# =========================================================================
# SECTION 2: IMPORTS + HELPERS
# =========================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

plt.rcParams["figure.dpi"] = 100
np.random.seed(EDA_CFG["random_state"])

CLASS_COLORS = {"MEL": "#d62728", "NV": "#7f7f7f", "BCC": "#2ca02c",
                "AKIEC": "#9467bd", "BKL": "#c49b9a", "DF": "#e377c2", "VASC": "#17becf"}


def onehot_to_label(df_raw):
    df = df_raw.copy()
    df["dx"] = df[CLASS_COLUMNS].idxmax(axis=1)
    return df[["image", "dx"]]


def load_gt(path):
    return onehot_to_label(pd.read_csv(path))


def stable_sample(files, n, seed):
    # Pilih n file secara deterministik (tidak acak-acakan antar run).
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(files), size=min(n, len(files)), replace=False)
    return [files[i] for i in sorted(idx)]


def dhash_pil(im):
    # Perceptual hash 64-bit: bandingkan kedekatan kecerahan piksel 9x8.
    gray = im.convert("L").resize((9, 8), Image.BILINEAR)
    px = list(gray.getdata())
    bits = 0
    for r in range(8):
        for c in range(8):
            bits = (bits << 1) | (1 if px[r * 9 + c] >= px[r * 9 + c + 1] else 0)
    return bits


def hamming(a, b):
    return bin(a ^ b).count("1")

## 3. EDA Tahap 1 — Distribusi Kelas

Hitung jumlah & persentase gambar per kelas, plus **rasio setiap kelas** terhadap kelas terkecil dan terbesar (bukan hanya minoritas). **Mengapa penting**: distribusi ini menjadi dasar penentuan `loss_weight`/balancing di pipeline. Dengan model berbasis CrossEntropy, kelas mayoritas (`NV`) bisa mendominasi gradien sedangkan kelas minoritas (`DF`, `VASC`) berisiko tidak terpelajari.

In [ ]:
# =========================================================================
# EDA TAHAP 1: Distribusi Kelas (training + test + validation resmi)
# =========================================================================
train_df = load_gt(TRAIN_GT_PATH)
test_df  = load_gt(TEST_GT_PATH)
val_df   = load_gt(VAL_GT_PATH)


def class_stats(df, name):
    cnt = df["dx"].value_counts().reindex(CLASS_COLUMNS).fillna(0).astype(int)
    pct = (cnt / cnt.sum() * 100).round(2)
    return pd.DataFrame({"class": CLASS_COLUMNS, name + "_count": cnt.values,
                         name + "_pct": pct.values})


dist = class_stats(train_df, "train")
dist = dist.merge(class_stats(test_df, "test"), on="class")
dist = dist.merge(class_stats(val_df, "val"), on="class")
print(dist.to_string(index=False))
dist.to_csv(OUTPUT_DIR / "eda_class_dist.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(dist["class"], dist["train_count"], color=[CLASS_COLORS[c] for c in dist["class"]])
for i, v in enumerate(dist["train_count"]):
    ax.text(i, v + 40, str(v), ha="center")
ax.set_title("EDA Tahap 1 — Distribusi Kelas (training set)")
ax.set_xlabel("Kelas"); ax.set_ylabel("Jumlah gambar"); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "eda_class_dist.png", dpi=120)
plt.show()

nv = dist.loc[dist["class"] == "NV", "train_count"].iloc[0]
df_ = dist.loc[dist["class"] == "DF", "train_count"].iloc[0]
print("Rasio NV : DF =", round(nv / df_, 1), ": 1")

# Rasio SETIAP kelas (terhadap kelas terkecil & terbesar), sesuai urutan CLASS_COLUMNS.
min_cnt = dist["train_count"].min()
max_cnt = dist["train_count"].max()
ratio_tbl = dist[["class", "train_count", "train_pct"]].copy()
ratio_tbl["rasio_vs_terkecil"] = (dist["train_count"] / min_cnt).round(2)
ratio_tbl["rasio_vs_terbesar"] = (max_cnt / dist["train_count"]).round(2)
print("\nRasio tiap kelas (jumlah gambar dibanding kelas terkecil / terbesar):")
print(ratio_tbl.to_string(index=False))
ratio_tbl.to_csv(OUTPUT_DIR / "eda_class_ratio.csv", index=False)
print("Setelah split val_ratio=", EDA_CFG["val_ratio"], ", minoritas bisa sangat tipis -> periksa EDA Tahap 7.")

## 4. EDA Tahap 2 — Analisis `lesion_id` (image_id vs lesion_id, risiko leakage)

**`HAM10000` tidak selalu berarti 1 image = 1 lesi independen.** Ada lesi yang difoto lebih dari satu kali sehingga punya beberapa `image_id` di bawah satu `lesion_id`:

```
lesion_id
   |--- image A
   |--- image B
   `--- image C
```

Konsekuensinya: jika pada split `image A -> train` dan `image B -> validation`, model bisa "mengintip" lesi yang sama di dua partition sekaligus → **data leakage** dan estimasi performa jadi **terlalu optimistis**. Implementasi HAM10000 modern secara eksplisit melakukan split berbasis `lesion_id` untuk mencegah hal ini.

Ground truth resmi ISIC 2018 tidak menyertakan `lesion_id`, sehingga analisis numerik menjalankan bila `HAM10000_metadata.csv` tersedia (otomatis dideteksi). Tanpa file itu, blok ini tetap menjelaskan konsep + cara melengkapi data.

In [ ]:
# =========================================================================
# EDA TAHAP 2: Analisis lesion_id (hanya bila HAM10000_metadata tersedia)
# =========================================================================
if HAM_META_PATH:
    meta = pd.read_csv(HAM_META_PATH)
    if "image_id" in meta.columns:
        meta = meta.rename(columns={"image_id": "image"})
    keep = [c for c in ["image", "lesion_id"] if c in meta.columns]
    if "lesion_id" not in meta.columns:
        print("HAM10000 metadata tidak punya kolom lesion_id -> abaikan kolom itu.")
    else:
        train_les = train_df.merge(meta[keep], on="image", how="left")
        has_les = train_les["lesion_id"].notna()
        print("image training yang punya lesion_id:", int(has_les.sum()),
              "dari", len(train_les))

        n_img = int(has_les.sum())
        n_les = train_les.loc[has_les, "lesion_id"].nunique()
        print("Jumlah image training (ber-lesion_id) :", n_img)
        print("Jumlah unique lesion_id               :", n_les)
        print("Rata-rata image per lesion            :", round(n_img / n_les, 3))

        per = train_les.loc[has_les].groupby("lesion_id")["image"].count()
        hist = per.value_counts().sort_index()
        print("\nDistribusi images per lesion:")
        print(hist.to_string())
        per.to_csv(OUTPUT_DIR / "eda_lesion_counts.csv", header=["n_images"])

        multi = per[per >= 2]
        print("\nJumlah lesion dengan >=2 gambar:", len(multi),
              "(%.1f%% dari lesion)" % (100.0 * len(multi) / len(per)))

        # 1) label dalam satu lesion harus konsisten
        same = train_les.loc[has_les].groupby("lesion_id")["dx"].nunique()
        print("Lesion dengan label tidak konsisten:", int((same > 1).sum()))

        # 2) contoh pohon (tree) lesi multi-gambar
        for lid in list(per[per >= 3].index[:2]) + list(per.index[:1]):
            rows = train_les[train_les["lesion_id"] == lid]
            imgs = list(rows["image"])
            lbl = rows["dx"].iloc[0]
            print("\nlesion_id=%s (label=%s, %d gambar)" % (lid, lbl, len(imgs)))
            for i, im in enumerate(imgs):
                branch = "`---" if i == len(imgs) - 1 else "+---"
                print("    ", branch, im)

        # 3) leakage: apakah lesi yang sama nyangkut di TRAIN-split DAN VAL-split?
        try:
            from sklearn.model_selection import train_test_split
            tr, va = train_test_split(train_les[has_les],
                                      test_size=EDA_CFG["val_ratio"],
                                      stratify=train_les.loc[has_les, "dx"],
                                      random_state=EDA_CFG["random_state"])
            les_map = train_les.loc[has_les].set_index("image")["lesion_id"].to_dict()
            tr_les = {les_map[i] for i in tr["image"] if i in les_map}
            va_les = {les_map[i] for i in va["image"] if i in les_map}
            overlap = sorted(tr_les & va_les)
            print("\n[LEAKAGE CHECK] Lesion yang muncul di TRAIN-split DAN VAL-split:",
                  len(overlap))
            if overlap:
                leak_rows = train_les[train_les["lesion_id"].isin(overlap)]
                leak_cnt = leak_rows.groupby("lesion_id")["image"].count().sort_values(ascending=False)
                print(leak_cnt.to_string())
                leak_rows.to_csv(OUTPUT_DIR / "eda_lesion_split_leak.csv", index=False)
                print("\nDaftar disimpan ke output_eda/eda_lesion_split_leak.csv")
                print("=> Pertimbangkan split berbasis lesion_id (lihat cell EDA Tahap 7).")
        except ImportError:
            print("\nsklearn tidak tersedia -> lewati cek leakage cross-split.")
else:
    print("HAM10000_metadata.csv TIDAK ditemukan.")
    print()
    print("Ground truth resmi ISIC 2018 Task 3 tidak menyertakan lesion_id.")
    print("Untuk analisis di atas (jumlah image, unique lesion_id, images per lesion,")
    print("dan cek leakage antar partition):")
    print("  1. Unduh file HAM10000_metadata.csv (missal dari HAM10000/ISIC Archive),")
    print("     lalu taruh di folder dataset/, ATAU")
    print("  2. Di Kaggle, attach dataset HAM10000 yang berisi file tsb (auto terdeteksi).")
    print("Tahap 2 tetap memberikan rekomendasi umum: split harus dijaga per lesion_id.")

## 5. EDA Tahap 3 — Visualisasi Setiap Kelas (2–3 gambar per kelas)

Tujuannya memahami pola visual pembeda antar kelas. Saat melihat grid di bawah, tanyakan:
- **shape**: bulat / tidak beraturan? simetris?
- **color**: merata atau sekadar beberapa warna (hitam, cokelat, merah)?
- **texture**: halus, granular, bersisik?
- **border**: tegas, bergigi, kabur?
- **size**: relatif terhadap frame, dipotong atau penuh?
- **global structure**: lesi kecil di tengah kulit vs lesi menutupi sebagian besar frame?

Catatan untuk observasi awal (bukan aturan mutlak): `MEL` sering warna campur & border tidak rata; `BKL` cenderung cokelat seragam; `BCC` sering mutiara/merah dengan pembuluh; `DF` cenderung keras & cokelat; `VASC` merah/pink.

In [ ]:
# =========================================================================
# EDA TAHAP 3: Grid sampel 3 gambar per kelas
# =========================================================================
files = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg")))
img_to_path = {os.path.splitext(os.path.basename(f))[0]: f for f in files}
by_dx = {cls: [img_to_path[i] for i in train_df[train_df["dx"] == cls]["image"] if i in img_to_path]
         for cls in CLASS_COLUMNS}

fig, axes = plt.subplots(len(CLASS_COLUMNS), 3, figsize=(9, 15))
for r, cls in enumerate(CLASS_COLUMNS):
    picks = stable_sample(by_dx[cls], 3, EDA_CFG["random_state"] + r)
    for c, f in enumerate(picks):
        im = Image.open(f)
        axes[r][c].imshow(im)
        axes[r][c].axis("off")
        axes[r][c].set_title(cls + " - " + os.path.splitext(os.path.basename(f))[0], fontsize=8)
fig.suptitle("EDA Tahap 3 — Sampel 3 gambar per kelas (training set)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(OUTPUT_DIR / "eda_class_samples_grid.png", dpi=110)
plt.show()

## 6. EDA Tahap 4 — Resolusi & Aspect Ratio

Periksa distribusi lebar, tinggi, dan aspect ratio sebelum memutuskan resize. Data ISIC asli umumnya bervariasi antara beberapa ratus piksel sampai ~1000 px, dan foto dermatoskop cenderung ~1:1 (lingkaran view) tetapi banyak juga yang crop persegi.

**Apakah harus resize 224×224?** Tidak otomatis harus, tapi ada konsekuensi langsung:
- **Computational cost & memory**: aktivasi & minibatch tumbuh kuadrat thd ukuran input.
- **Detail lesi**: resize mengecilkan bisa menghilangkan tekstur halus / border bergerigi (penting utk MEL vs BKL).
- **Receptive field**: struktur ResNet dirancang utk input 224; di input lain, field efektif layer konvolusi berubah relatif thd ukuran objek.
- **Batch size**: input lebih besar memaksa batch kecil (stabilitas optimasi berubah).

Bila aspek ratio tidak 1:1, resize 224×224 akan mendistorsi — opsi umum: resize dengan pisah lurus (mempertahankan ratio) lalu center-crop / pad.

In [ ]:
# =========================================================================
# EDA TAHAP 4: Resolusi & aspect ratio (sampel)
# =========================================================================
sample_files = stable_sample(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg")),
                             EDA_CFG["sample_size"], EDA_CFG["random_state"])
rows = []
for f in sample_files:
    with Image.open(f) as im:
        w, h = im.size
    rows.append({"image": os.path.basename(f), "width": w, "height": h})
res = pd.DataFrame(rows)
res["aspect_ratio"] = (res["width"] / res["height"]).round(3)

print("Statistik resolusi (sampel n=%d):" % len(res))
print(res[["width", "height", "aspect_ratio"]].agg(["mean", "median", "std", "min", "max"]).round(2).to_string())
res.to_csv(OUTPUT_DIR / "eda_resolution_stats.csv", index=False)

fig, axs = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col, ttl in zip(axs, ["width", "height", "aspect_ratio"],
                        ["Lebar (px)", "Tinggi (px)", "Aspect ratio (w/h)"]):
    ax.hist(res[col], bins=30, color="#1f77b4", edgecolor="white")
    ax.set_title(ttl)
    ax.set_xlabel("px" if col != "aspect_ratio" else "nilai")
axs[0].set_ylabel("jumlah gambar")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "eda_resolution_hists.png", dpi=120)
plt.show()

# ---- estimasi kebutuhan memori input (RGB float32) ----
print("Estimasi memori batch utk input RGB float32:")
for s in [128, 224, 299]:
    b_img = 3 * s * s * 4
    print("  input %3dx%-3d : %6d KB/gambar | batch 32 = %6.1f MB" % (s, s, b_img // 1024, b_img * 32 / 1024 ** 2))

## 7. EDA Tahap 5 — Distribusi Warna (RGB histogram, brightness, contrast, saturation)

Warna bisa menjadi fitur diskriminatif kuat pada lesi kulit (mis. `VASC` merah, `DF` cokelat, `MEL` campuran). Karena itu keputusan augmentasi **ColorJitter tidak boleh sembarangan**:

```
lesion asli --augmentasi--> warna berubah ekstrem
model mungkin belajar distribusi warna yang tidak realistis
```

Setelah melihat distribusi statistik di bawah, kita bisa menilai: seberapa jauh `ColorJitter` (mis. brightness/saturation/contrast ±0.2) menggeser statistik distribusi training. Jika rentang asli sempit dan hasil augmentasi jauh menyeberang rentang itu, kuatkan jitter; jika menggeser ekstrem ke area yang tidak wajar, lemahkan.

In [ ]:
# =========================================================================
# EDA TAHAP 5: RGB histogram + brightness/contrast/saturation (sampel)
# =========================================================================
sample_files = stable_sample(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg")),
                             EDA_CFG["color_sample_size"], 7)
means = np.zeros(len(sample_files))
vals = {"R": [], "G": [], "B": [], "brightness": [], "contrast": [], "saturation": []}
hist_acc = {c: None for c in "RGB"}

for f in sample_files:
    im = Image.open(f).convert("RGB")
    arr = np.asarray(im).astype(np.float32)
    lum = arr @ np.array([0.299, 0.587, 0.114])
    hsv = np.asarray(im.convert("HSV")).astype(np.float32)
    vals["R"].append(arr[..., 0].mean()); vals["G"].append(arr[..., 1].mean())
    vals["B"].append(arr[..., 2].mean())
    vals["brightness"].append(lum.mean()); vals["contrast"].append(lum.std())
    vals["saturation"].append(hsv[..., 1].mean() / 255.0)
    small = arr[::4, ::4].astype(np.uint8)
    for ci, c in enumerate("RGB"):
        h, _ = np.histogram(small[..., ci].ravel(), bins=64, range=(0, 256))
        hist_acc[c] = h / h.sum() if hist_acc[c] is None else hist_acc[c] + h / h.sum()
    im.close()

for c in "RGB":
    hist_acc[c] /= len(sample_files)

color_stats = pd.DataFrame({
    "stat": ["mean_R", "mean_G", "mean_B", "brightness", "contrast", "saturation"],
    "mean":  [np.mean(vals["R"]), np.mean(vals["G"]), np.mean(vals["B"]),
              np.mean(vals["brightness"]), np.mean(vals["contrast"]), np.mean(vals["saturation"])],
    "median": [np.median(vals["R"]), np.median(vals["G"]), np.median(vals["B"]),
               np.median(vals["brightness"]), np.median(vals["contrast"]), np.median(vals["saturation"])],
    "std":   [np.std(vals["R"]), np.std(vals["G"]), np.std(vals["B"]),
              np.std(vals["brightness"]), np.std(vals["contrast"]), np.std(vals["saturation"])],
}).round(2)
print(color_stats.to_string(index=False))
color_stats.to_csv(OUTPUT_DIR / "eda_color_stats.csv", index=False)

chan_col = {"R": "red", "G": "green", "B": "blue"}
fig = plt.figure(figsize=(13, 6.5))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1])
ax0 = fig.add_subplot(gs[0, :])
for c in "RGB":
    bin_c = np.linspace(4, 252, 64)
    ax0.plot(bin_c, hist_acc[c], color=chan_col[c], label=c)
ax0.set_title("Rata-rata histogram RGB (sampel)")
ax0.set_xlabel("intensitas"); ax0.set_ylabel("fraksi piksel"); ax0.legend(); ax0.grid(alpha=0.3)

for i, (key, ttl) in enumerate([("brightness", "Brightness"), ("contrast", "Contrast (std lum)"),
                                ("saturation", "Saturation (HSV)")]):
    axb = fig.add_subplot(gs[1, i])
    axb.hist(vals[key], bins=30, color="#9467bd", edgecolor="white")
    axb.set_title(ttl); axb.set_xlabel("nilai rata-rata per gambar")
axb.set_ylabel("jumlah gambar")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "eda_color_hists.png", dpi=120)
plt.show()

### 7.1 Sanity-check ColorJitter (berapa jauh augmentasi menggeser distribusi?)

Simulasi `ColorJitter(brightness=0.2, saturation=0.2, contrast=0.2)` memakai `PIL.ImageEnhance` (faktor acak per gambar, seed tetap). Bandingkan statistik asli vs setelah jitter: kita ingin jitter memperluas distribusi **secukupnya** — tidak terlalu sempit (overfit) dan tidak terlalu ekstrem (membuat warna tidak realistis).

In [ ]:
# =========================================================================
# EDA TAHAP 5b: Simulasi ColorJitter (PIL ImageEnhance) & bandingkan statistik
# =========================================================================
jb, js_, jc = 0.2, 0.2, 0.2
rng = np.random.RandomState(11)
orig, jitt = {"brightness": [], "contrast": [], "saturation": []},              {"brightness": [], "contrast": [], "saturation": []}
hue_shifted = []

for f in sample_files:
    im = Image.open(f).convert("RGB")
    lum = np.asarray(im).astype(np.float32) @ np.array([0.299, 0.587, 0.114])
    sat0 = np.asarray(im.convert("HSV")).astype(np.float32)[..., 1].mean() / 255.0
    orig["brightness"].append(lum.mean())
    orig["contrast"].append(lum.std())
    orig["saturation"].append(sat0)

    i2 = ImageEnhance.Brightness(im).enhance(1 + rng.uniform(-jb, jb))
    i2 = ImageEnhance.Color(i2).enhance(1 + rng.uniform(-js_, js_))
    i2 = ImageEnhance.Contrast(i2).enhance(1 + rng.uniform(-jc, jc))
    lum2 = np.asarray(i2).astype(np.float32) @ np.array([0.299, 0.587, 0.114])
    sat2 = np.asarray(i2.convert("HSV")).astype(np.float32)[..., 1].mean() / 255.0
    jitt["brightness"].append(lum2.mean())
    jitt["contrast"].append(lum2.std())
    jitt["saturation"].append(sat2)
    im.close()

rows = []
for k in orig:
    rows.append({"stat": k,
                 "orig_mean": np.mean(orig[k]), "jitter_mean": np.mean(jitt[k]),
                 "orig_median": np.median(orig[k]), "jitter_median": np.median(jitt[k]),
                 "orig_std": np.std(orig[k]), "jitter_std": np.std(jitt[k])})
jdf = pd.DataFrame(rows).round(3)
print(jdf.to_string(index=False))
jdf.to_csv(OUTPUT_DIR / "eda_jitter_shift.csv", index=False)

fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for ax, k in zip(axs, ["brightness", "contrast", "saturation"]):
    ax.boxplot([orig[k], jitt[k]])
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["asli", "jitter +-0.2"])
    ax.set_title(k)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "eda_jitter_shift.png", dpi=120)
plt.show()

print("Bacaan: 'jitter_std' yg jauh lebih besar dari 'orig_std' = augmentasi memperlebar",
      "distribusi; jika 'jitter_median' keluar dari rentang orig, warna tidak realistis.")

## 8. EDA Tahap 6 — Duplikat / Near-Duplikat

Selain `lesion_id`, perlu juga memeriksa gambar yang **sama persis** (md5) atau **hampir sama** (dhash, 64-bit) karena: `train ~= validation` membuat evaluasi **bias**. Deteksi dilakukan antar partition: training (bisa di-split lagi), test resmi, validation resmi.

- **Exact duplicate**: md5 file. Jika `full_hash=True` (default), seluruh gambar pada ketiga partition di-hash (bisa butuh 1–2 menit).
- **Near duplicate**: perbandingan `dhash` berpasangan pada sampel (`EDA_CFG['near_dup_sample']`) dengan ambang hamming `EDA_CFG['ham_near_threshold']`.

In [ ]:
# =========================================================================
# EDA TAHAP 6: Exact duplicate (md5) seluruh partition + near dup (dhash)
# =========================================================================

# ---- 6a. exact duplicate ----
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

print("Hash md5 (exact duplicate)...")
hash_rows = []
for part, imgdir in [("train", TRAIN_IMG_DIR), ("test", TEST_IMG_DIR), ("val", VAL_IMG_DIR)]:
    fl = sorted(glob.glob(os.path.join(imgdir, "*.jpg")))
    if not EDA_CFG["full_hash"]:
        fl = stable_sample(fl, EDA_CFG["near_dup_sample"], 3)
    for f in fl:
        hash_rows.append({"image": os.path.basename(f), "split": part, "md5": file_md5(f)})
hdf = pd.DataFrame(hash_rows)
hdf.to_csv(OUTPUT_DIR / "eda_dup_exact.csv", index=False)

dups = hdf[hdf.duplicated("md5", keep=False)].sort_values(["md5", "split"])
print("Jumlah entry gambar duplikat eksak (md5 sama):", len(dups))
if len(dups):
    for md5v, g in dups.groupby("md5"):
        print("  ", md5v[:8], "->", list(zip(g["image"].tolist(), g["split"].tolist())))

tr_h = set(hdf.loc[hdf["split"] == "train", "md5"])
te_h = set(hdf.loc[hdf["split"] == "test", "md5"])
va_h = set(hdf.loc[hdf["split"] == "val", "md5"])
print("Duplikat eksak training vs test :", len(tr_h & te_h))
print("Duplikat eksak training vs val  :", len(tr_h & va_h))

# ---- 6b. near duplicate (dhash) pada sampel training ----
print("\nHitung dhash near-duplicate pada sampel", EDA_CFG["near_dup_sample"], "gambar...")
sam = stable_sample(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg")),
                    EDA_CFG["near_dup_sample"], 5)
sh = [(os.path.basename(f), dhash_pil(Image.open(f))) for f in sam]
pairs = []
for i in range(len(sh)):
    for j in range(i + 1, len(sh)):
        d = hamming(sh[i][1], sh[j][1])
        if d <= EDA_CFG["ham_near_threshold"]:
            pairs.append((sh[i][0], sh[j][0], d))
pairs.sort(key=lambda x: x[2])
near = pd.DataFrame(pairs, columns=["image_a", "image_b", "hamming"])
near.to_csv(OUTPUT_DIR / "eda_dup_near.csv", index=False)
print("Pasangan near-duplicate (hamming <= %d): %d" % (EDA_CFG["ham_near_threshold"], len(near)))
if len(near):
    print(near.head(20).to_string(index=False))

## 9. EDA Tahap 7 — Ukuran Dataset Setelah Split (per kelas)

Setelah split (sama dengan pipeline utama: stratified `train`/`val` dari training set resmi, `test` = test set resmi), lihat jumlah **per kelas** — bukan hanya total gambar:

```
       train  val  test
akiec    ...   ...   ...
bcc      ...   ...   ...
bkl      ...   ...   ...
df       ...   ...   ...   <- jauh lebih sedikit
mel      ...   ...   ...
nv       ...   ...   ...
vasc     ...   ...   ...   <- jauh lebih sedikit
```

`DF` dan `VASC` yang sangat jarang akan mempengaruhi:
- **class weight** (bobot CrossEntropy — pipeline memakai `inverse_frequency`),
- **sampling** (balancing/oversampling),
- **augmentation** (berapa kuat utk kelas minoritas),
- **interpretasi F1**: accuracy oval tapi F1 per kelas rendah = mayoritas mendominasi,
- **reliabilitas metrik per kelas**: dengan n sangat kecil, CI lebar & fluktuasi besar.

Bila `HAM10000_metadata` tersedia, bagian kedua menampilkan perbandingan split biasa vs **split berbasis `lesion_id`** (anti-leakage) beserta cek lesi lintas partition.

In [ ]:
# =========================================================================
# EDA TAHAP 7: Jumlah per kelas per partition + opsi split berbasis lesion
# =========================================================================
try:
    from sklearn.model_selection import train_test_split
except ImportError:
    train_test_split = None

if train_test_split is None:
    print("scikit-learn tidak terinstall -> bagian split (Tahap 7) dilewati.")
    print("Install scikit-learn (Kaggle/Colab sudah tersedia) untuk menghitung train/val/test per kelas.")
else:
    tr_imgs, va_imgs = train_test_split(train_df, test_size=EDA_CFG["val_ratio"],
                                        stratify=train_df["dx"],
                                        random_state=EDA_CFG["random_state"])

    def counts_of(df):
        return df["dx"].value_counts().reindex(CLASS_COLUMNS).fillna(0).astype(int)

    table = pd.DataFrame({
        "train": counts_of(tr_imgs),
        "val": counts_of(va_imgs),
        "test": counts_of(test_df),
    })
    table["train_%"] = (table["train"] / table["train"].sum() * 100).round(1)
    table["val_%"] = (table["val"] / table["val"].sum() * 100).round(1)
    table["test_%"] = (table["test"] / table["test"].sum() * 100).round(1)
    print(table.to_string())
    table.to_csv(OUTPUT_DIR / "eda_split_counts.csv")

    table[["train", "val", "test"]].plot(kind="bar", figsize=(10, 4.5), rot=0, colormap="viridis")
    plt.title("EDA Tahap 7 — Jumlah gambar per kelas per partition")
    plt.xlabel("Kelas"); plt.ylabel("Jumlah gambar"); plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eda_split_counts.png", dpi=120)
    plt.show()

    print("Kelas minoritas (DF, VASC) di validation split berturut-turut:",
          table.loc["DF", "val"], table.loc["VASC", "val"], "gambar")
    print("Kelas minoritas di test resmi:", table.loc["DF", "test"], table.loc["VASC", "test"])

    # ---- Opsional: split berbasis lesion_id (anti-leakage) ----
    if HAM_META_PATH and "train_les" in dir() and train_les["lesion_id"].notna().any():
        les = train_les.dropna(subset=["lesion_id"]).copy()
        # grup image per lesion; label = label terbanyak dalam lesi (harusnya konsisten, sudah dicek)
        g = les.groupby("lesion_id").agg({"dx": lambda s: s.mode().iloc[0], "image": "count"})
        g = g.rename(columns={"image": "n_img"})
        tr_l, va_l = train_test_split(g.reset_index(), test_size=EDA_CFG["val_ratio"],
                                      stratify=g["dx"], random_state=EDA_CFG["random_state"])
        tr_l_img = set(tr_l["lesion_id"]); va_l_img = set(va_l["lesion_id"])
        print("\n[LESION-AWARE SPLIT] lesion di train-split:", len(tr_l_img),
              "| lesion di val-split:", len(va_l_img),
              "| overlap (harusnya 0):", len(tr_l_img & va_l_img))
        tr_no = les[les["lesion_id"].isin(tr_l_img)]
        va_no = les[les["lesion_id"].isin(va_l_img)]
        t2 = pd.DataFrame({"train": counts_of(tr_no), "val": counts_of(va_no)})
        t2["train_%"] = (t2["train"] / t2["train"].sum() * 100).round(1)
        t2["val_%"] = (t2["val"] / t2["val"].sum() * 100).round(1)
        print(t2.to_string())
        print("=> Semua image dari lesion yang sama berada di partition yang sama (zero leakage).")

## 10. Ringkasan & Implikasi

Isi kolom `...`/angka sesuai hasil eksekusi di atas (contoh template):

1. **Ketimpangan kelas**: `NV` dominan; `DF` & `VASC` paling sedikit. → pipeline wajib packai `loss_weight`/balancing; F1 per kelas lebih informatif daripada accuracy.
2. **`lesion_id`**: ada lesi dengan beberapa gambar (`image A/B/C`). Split acak berisiko menempatkan lesi yang sama di train & val → **bias optimistis**. Solusi: split berbasis lesion_id bila metadata tersedia. *(Sertakan angka dari EDA tahap 2.)*
3. **Visi per kelas**: catat pembeda visual yang paling mencolok per kelas (shape/color/texture/border/size).
4. **Resolusi**: mayoritas gambar sekitar X px, aspect ratio Y. Resize `{img_size}` pertimbangkan detail lesi vs memori/batch.
5. **Warna**: rentang brightness/contrast/saturation; hasil jitter ±0.2 menggeser statistik sebesar Z → pertimbangkan kekuatan ColorJitter.
6. **Duplikat**: ada/tidak exact & near duplicate, termasuk antar partition (jumlah pasangan). Bila ada, eksklusi dari split.
7. **Split**: tabel per kelas di atas jadi dasar penutupan modell & interpretasi metrik.

Semua artefak tersimpan di `output_eda/` (CSV + PNG).